# Section A2: Image Classification (Fashion-MNIST with MobileNetV2)

This notebook trains a product category image classifier using **Fashion-MNIST** and **Transfer Learning with MobileNetV2** in TensorFlow / Keras.

### Pipeline Overview:
1. **Load Dataset**: Directly from `tf.keras.datasets.fashion_mnist` (60,000 train, 10,000 test).
2. **EDA**: Inspect shapes, class distribution, pixel ranges, and visualize sample images.
3. **Preprocessing**: Convert 28x28 grayscale to 3-channel RGB, resize for MobileNetV2 input, and apply `mobilenet_v2.preprocess_input`.
4. **Train / Validation Split**: 80% Training (48,000), 20% Validation (12,000), 10,000 Test.
5. **Data Augmentation**: Small rotations (±10°), zooms, and subtle width/height shifts.
6. **Model Architecture**: Transfer learning using pretrained MobileNetV2 base + GAP + Dropout + Dense classification head.
7. **Model Training**: Transfer learning phase (frozen base) + Fine-tuning phase.
8. **Evaluation**: Test accuracy, classification report, confusion matrix, and prediction output (predicted category + confidence score).
9. **Save Artifact**: Export model to `product_classifier.h5`.

### Step 1 — Load Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Load Fashion-MNIST directly from tf.keras.datasets
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

# Define class labels mapping
class_names = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

print(f"Loaded Training Images: {X_train_full.shape}")
print(f"Loaded Testing Images: {X_test.shape}")

### Step 2 — Exploratory Data Analysis (EDA)

In [ ]:
print("=== Dataset Overview ===")
print("Training Images Count:", X_train_full.shape[0])
print("Testing Images Count:", X_test.shape[0])
print("Image Dimensions:", X_train_full.shape[1:], "(Grayscale)")
print("Number of Classes:", len(class_names))
print("Pixel Value Range: Min =", X_train_full.min(), ", Max =", X_train_full.max())

# Check Class Balance
class_counts = pd.Series(y_train_full).value_counts().sort_index()
print("\nImages per Class (Training):")
for idx, count in class_counts.items():
    print(f"  Class {idx} ({class_names[idx]}): {count}")

# Display grid of sample images with labels
plt.figure(figsize=(12, 6))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_train_full[i], cmap='gray')
    plt.title(f"{class_names[y_train_full[i]]} ({y_train_full[i]})")
    plt.axis('off')
plt.tight_layout()
plt.show()

### Step 3 — Preprocessing

- MobileNetV2 expects input shape: `224 × 224 × 3` (or minimum 32x32x3 for transfer learning).
- We stack the grayscale single channel to 3 identical RGB channels.
- Resize images to target size for MobileNetV2 feature extraction.
- Apply `mobilenet_v2.preprocess_input()`.

In [ ]:
# Function to preprocess 28x28 grayscale images for MobileNetV2
# We resize images (e.g. to 96x96 or 224x224) and repeat 1 channel to 3 channels
IMG_SIZE = (96, 96) # 96x96 provides excellent spatial resolution for MobileNetV2 with fast execution

def preprocess_images(images):
    # Add channel dimension (N, 28, 28, 1)
    images_expanded = np.expand_dims(images, axis=-1)
    # Repeat grayscale to 3 channels (N, 28, 28, 3)
    images_rgb = np.repeat(images_expanded, 3, axis=-1)
    return images_rgb

# Convert datasets to 3-channel RGB
X_train_rgb = preprocess_images(X_train_full)
X_test_rgb = preprocess_images(X_test)

print("Processed Training RGB Shape:", X_train_rgb.shape)
print("Processed Testing RGB Shape:", X_test_rgb.shape)

### Step 4 — Train / Validation Split

In [ ]:
# Split training set into 80% Train and 20% Validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_rgb, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)

print(f"Training set:   {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Testing set:    {X_test_rgb.shape[0]} samples")

### Step 5 — Data Augmentation Pipeline

In [ ]:
# Build sequential Data Augmentation and MobileNetV2 preprocessing layer
data_augmentation = tf.keras.Sequential([
    layers.Resizing(IMG_SIZE[0], IMG_SIZE[1]),
    layers.RandomRotation(0.05), # ±10 degrees
    layers.RandomZoom(0.05),
    layers.RandomTranslation(height_factor=0.05, width_factor=0.05),
    layers.Lambda(lambda x: preprocess_input(x))
], name="data_augmentation_and_preprocessing")

# Validation & Test preprocessing layer without augmentation
val_test_preprocessing = tf.keras.Sequential([
    layers.Resizing(IMG_SIZE[0], IMG_SIZE[1]),
    layers.Lambda(lambda x: preprocess_input(x))
], name="val_test_preprocessing")

### Step 6 — Build the Model (MobileNetV2 Transfer Learning)

In [ ]:
# Load pretrained MobileNetV2 base without top classification layer
base_model = MobileNetV2(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    include_top=False,
    weights='imagenet'
)

# Freeze MobileNetV2 feature extractor
base_model.trainable = False

# Build full architecture
inputs = layers.Input(shape=(28, 28, 3), name="input_image")
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
x = layers.Dropout(0.3, name="dropout")(x)
x = layers.Dense(128, activation='relu', name="dense_128")(x)
outputs = layers.Dense(10, activation='softmax', name="output_softmax")(x)

model = models.Model(inputs, outputs, name="FashionMNIST_MobileNetV2")
model.summary()

### Step 7 — Train Model

- Optimizer: `Adam(learning_rate=1e-3)`
- Loss: `SparseCategoricalCrossentropy()`
- Metrics: `accuracy`

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

# Define Callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=3, restore_best_weights=True
)

EPOCHS = 10
BATCH_SIZE = 64

print("=== Starting Phase 1 Training (Classifier Head) ===")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping]
)

### Step 8 — Evaluation & Prediction (Category + Confidence Score)

In [ ]:
# Preprocess test dataset using evaluation pipeline
test_preds_prob = model.predict(X_test_rgb, batch_size=BATCH_SIZE)
y_pred = np.argmax(test_preds_prob, axis=1)
confidence_scores = np.max(test_preds_prob, axis=1)

test_acc = accuracy_score(y_test, y_pred)
print(f"=== Final Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%) ===\n")

print("Classification Report:\n", classification_report(y_test, y_pred, target_names=class_names))

# Confusion Matrix Visualization
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Category')
plt.ylabel('True Category')
plt.title('Fashion-MNIST Confusion Matrix')
plt.show()

# Sample Prediction Output (Predicted Category + Confidence Score)
print("\n=== Sample Predictions Output ===")
for idx in range(5):
    pred_class = class_names[y_pred[idx]]
    true_class = class_names[y_test[idx]]
    conf = confidence_scores[idx] * 100
    print(f"Sample {idx+1}: Predicted = '{pred_class}' ({conf:.2f}% confidence) | Actual = '{true_class}'")

### Step 9 — Save Model Artifact (`product_classifier.h5`)

In [ ]:
# Save model deliverable
model_path = 'product_classifier.h5'
model.save(model_path)
print(f"Saved product classifier deliverable model successfully at '{model_path}'!")